In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, read_in_routine, get_metadata
from plot_icusleep_data import icu_sleep_plot
from scipy.stats import variation
from plotly.offline import plot
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
import re

%load_ext autoreload
%autoreload 2

In [2]:
# import time
# time.sleep(60*15)

In [3]:
cohort = 'icu'
data_dir = f'/media/ssd2/{cohort}_final_files'
data_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step_5_ecg_sleepstaging_nohrv/'
data_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step3_nohrv'

# plots_dir = f'/media/mad3/{cohort}_plots_sleep_stages'
plots_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleep_staging/plots_09_03'

# cohort = 'covid_resp'
# # data_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2'
# data_dir = '/media/ssd2/Combined_Data5'

# plots_dir = f'/media/mad3/Projects/covid_data/{cohort}_plots_sleep_stages_comb5'
# # plots_dir = '/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2_Plots'



if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)

files = os.listdir(data_dir)
files.sort()
len(files)

130

In [4]:
# # only select the top10 rem patients:

# top10rem = pd.read_csv('top10_rem.csv')
# top10rem = top10rem.study_id.apply(lambda x: x.split('_')[1]).values
# # files
# files = [x for x in files if np.any([y in x for y in top10rem])]
# print(files)

In [5]:
# signals_to_plot_today = [
#                     'acc_ai_10sec',
#                    'apnea_pred_ro_a3',
#                    'apnea_pred_rsr_a3',
#                    'movavg_0_5s_unscaled', 
#                    'rr_10s_smooth', 
#                    'edw_respirations',
#                    'ventilation_10s_smooth',
#                    'instability_index_30sec',
#                    'instability_index_2min',
#                     'stage_pred_amendsumeffort',   
#                     'stage_pred_activity10sec',
#                     'stage_pred_ecg_nn',
#                          'nn_interval',
#                   'hr', 'spo2', 'art1s', 'art1d', 'nbps', 'nbpd', 'resp', 'vent rate',
#                          'self_similarity',
#                          'edw_bp_diastolic',
#                          'edw_bp_systolic',
#                          'edw_pulse',
#                          'edw_pulse_oximetry',
#                   'oxygen_flow_rate', 'oxygen_device',
#     'positioned', 
#     'airgo_signal_quality', 
#     'hypoxic_area_va_a3',
#     'cpc_hfc_lfc_ratio',
#                         'sofa_respiratory', 'sofa_cvs', 'sofa_score', 
#     'opioids_sum', 'benzos_sum', 'antipsychotics_sum',
#                         ]

In [6]:
signals_to_plot_today = [
                    'acc_ai_10sec',
                   'apnea_pred_ro_a3',
                   'apnea_pred_rsr_a3',
                   'movavg_0_5s_unscaled', 
                   'rr_10s_smooth', 
                   'edw_respirations',
                   'ventilation_10s_smooth',
                   'instability_index_30sec',
                   'instability_index_2min',
                    'stage_pred_amendsumeffort',   
                    'stage_pred_activity10sec',
                  'hr', 'spo2', 'art1s', 'art1d', 'nbps', 'nbpd', 'resp', 'vent rate',
                         'self_similarity',
                         'edw_bp_diastolic',
                         'edw_bp_systolic',
                         'edw_pulse',
                         'edw_pulse_oximetry',
                  'oxygen_flow_rate', 'oxygen_device',
    'positioned', 
    'hypoxic_area_va_a3',
                        'sofa_respiratory', 'sofa_cvs', 'sofa_score', 
    'opioids_sum', 'benzos_sum', 'antipsychotics_sum',
                        ]


signals_to_plot_full = ['acc_ai_10sec',
                   'apnea_pred_ro_a3',
                   'apnea_pred_rsr_a3',
                   'movavg_0_5s_unscaled', 
                   'rr_10s_smooth', 
                   'edw_respirations',
                   'ventilation_10s_smooth',
                   'instability_index_30sec',
                   'instability_index_2min',
                   'stage_pred_comb_breath_activity_1',
                  'stage_pred_amendsumeffort',   
                    'stage_pred_activity10sec',       
                  'hr', 'spo2', 'art1s', 'art1d', 'nbps', 'nbpd', 'resp', 'vent rate',
                  'oxygen_flow_rate', 'oxygen_device', 'positioned', 'airgo_signal_quality']


signals_to_plot_all = ['acc_ai_10sec',
                   'apnea_pred_ro_a3',
                   'apnea_pred_rsr_a3',
                   'movavg_0_5s_unscaled', 
                        'self_similarity',
                        'hypoxic_area_va_a3',
                   'rr_10s_smooth', 
                   'edw_respirations',
                   'ventilation_10s_smooth',
                   'instability_index_30sec',
                   'instability_index_2min',
                       'airgo_signal_quality',
                   'stage_pred_comb_breath_activity_1',
                  'stage_pred_amendsumeffort',   
                    'stage_pred_activity10sec',       
                  'hr', 'spo2', 'art1s', 'art1d', 'nbps', 'nbpd', 'resp', 'vent rate',
                  'oxygen_flow_rate', 'oxygen_device', 'positioned']


signals_to_plot_sub = ['acc_ai_10sec',
                   'movavg_0_5s_unscaled', 
                   'rr_10s_smooth', 
                   'edw_respirations',
                   'instability_index_2min',
                   'stage_pred_comb_breath_activity_1',
                  'hr', 'spo2', 'resp', 'vent rate',
                  'oxygen_flow_rate', 'oxygen_device', 'positioned', 'airgo_signal_quality']

signals_to_plot_resp = ['movavg_0_5s_unscaled', 
                   'instability_index_2min',
                   'stage_pred_comb_breath_activity_1',
                  'oxygen_flow_rate', 'oxygen_device', 'resp', 'vent rate', 'rr_10s_smooth',  'edw_respirations', 'airgo_signal_quality']

In [7]:
def plot_routine(data):
    title = study_id
    hours = data.shape[0]/hdr['fs']/3600
    parts = int(np.ceil(hours*len(signals_available_plot)/2500))  # 850
    len_part = data.shape[0]//parts
    print(parts)
    print(len_part/fs/3600)


    for part in range(parts):
    
        part_start_dt = data.iloc[[int(len_part*part)]].index.values[0]
        try:
            part_end_dt = data.iloc[[int(len_part*(part+1)+1)]].index.values[0]
        except:
            part_end_dt = None

        fig = icu_sleep_plot(data.loc[part_start_dt : part_end_dt, signals_available_plot], 
                             trace_linewidth=1, 
                             labelfontsize=12, legend_position = [0.2, 1.1], legend_font_size=12)
        fig.update_layout(title=title)
        plot(fig, filename = os.path.join(plots_dir, title+f'{version}_part{part+1}.html'), auto_open=False) # save_path.replace('FILEFORMAT','HTML')+'.html'
        

In [8]:
# data_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_v2' # _hrv'
# files = os.listdir(data_dir)
# files.sort()
# file= files[70]
# print(file)

In [9]:
# import re

# data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
# fs = hdr['fs']


In [10]:
# study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
# version = 'today'
# fs = hdr['fs']
# signals_available_plot = [x for x in signals_to_plot_today if x in data.columns]
# plot_routine()

In [14]:
files[90]

'icusleep_135.h5'

In [16]:
for file in files[90:]:
    
    try:

        print(file)
        data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
        data.rename({'stage_pred_vcomb1': 'stage_pred_comb_breath_activity_1'}, axis=1, inplace=True)

        if 1:

            data['positioned'] = data['positioning_frequency'].astype(str) + ', ' + data['repositioned'].astype(str)
            data['positioned'] = data.positioned.apply(lambda x: x.replace('nan, nan','nan'))

            study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
            fs = hdr['fs']

            version = 'today'
            signals_available_plot = [x for x in signals_to_plot_today if x in data.columns]
            signals_available_plot = [x for x in signals_available_plot if data[x].dropna().shape[0]>0]

            plot_routine(data)

    #         version = 'all'
    #         signals_available_plot = [x for x in signals_to_plot_all if x in data.columns]
    #         plot_routine()

    #         version = 'full'
    #         signals_available_plot = [x for x in signals_to_plot_full if x in data.columns]
    #         plot_routine()

    #         version = 'sub'
    #         signals_available_plot = [x for x in signals_to_plot_sub if x in data.columns]    
    #         plot_routine()

    #         version = 'resp'
    #         signals_available_plot = [x for x in signals_to_plot_resp if x in data.columns]    
    #         plot_routine()    
    
    except Exception as e:
        print(e)
        continue

icusleep_135.h5
2
72.01666666666667
icusleep_137.h5
6
71.16388888888889
icusleep_139.h5
2
59.233333333333334
icusleep_140.h5
2
49.642694444444444
icusleep_142.h5
1
49.76669444444445
icusleep_143.h5
1
76.23336111111111
icusleep_145.h5
2
61.61666666666667
icusleep_149.h5
1
46.63447222222222
'DataFrame' object has no attribute 'oxygen_flow_rate'
icusleep_151.h5
1
74.18336111111111
icusleep_152.h5
1
50.58336111111111
icusleep_153.h5
1
97.26808333333332
'DataFrame' object has no attribute 'oxygen_flow_rate'
icusleep_154.h5
2
48.125
icusleep_155.h5
3
78.29088888888889
'DataFrame' object has no attribute 'oxygen_flow_rate'
icusleep_156.h5
1
83.99363888888888
'DataFrame' object has no attribute 'oxygen_flow_rate'
icusleep_157.h5
1
75.90002777777777
icusleep_158.h5
2
55.675
icusleep_160.h5
1
53.25002777777778
icusleep_161.h5
2
60.572916666666664
icusleep_164.h5
1
24.466694444444446
icusleep_165.h5
1
85.71669444444444
icusleep_167.h5
2
49.90230555555555
icusleep_168.h5
4
67.20833333333333
icusle

In [13]:
data.hypoxic_area_va_a3.sum()

10804.111

In [27]:
plot_routine(data)

3
45.72313888888888


In [21]:
data.nn_interval.dropna()

2019-05-30 21:14:13.600    0.560
2019-05-30 21:14:14.700    1.055
2019-05-30 21:14:31.900    1.150
2019-05-30 21:14:32.400    0.470
2019-05-30 21:14:33.100    0.785
                           ...  
2019-06-05 11:12:57.900    0.655
2019-06-05 11:12:58.600    0.650
2019-06-05 11:12:59.200    0.650
2019-06-05 11:12:59.900    0.650
2019-06-05 11:13:12.500    0.720
Name: nn_interval, Length: 677239, dtype: float32

In [58]:
print(f"Apnea:                  {'apnea_pred_va_a3' in data.columns}")
print(f"Sleep Stage Breathing:  {'stage_pred_amendsumeffort' in data.columns}")
print(f"Sleep Stage Activity:   {'stage_pred_activity10sec' in data.columns}")
print(f"Sleep Stage Comb:       {'stage_pred_comb_breath_activity_1' in data.columns}")
print(f"Self Similarity:        {'self_similarity' in data.columns}")
print(f"Hypoxic Burden:         {'hypoxic_area' in data.columns}")
print(f"Oxygen Flow:            {'oxygen_flow_rate' in data.columns}")

Apnea:                  True
Sleep Stage Breathing:  True
Sleep Stage Activity:   True
Sleep Stage Comb:       True
Self Similarity:        True
Hypoxic Burden:         True
Oxygen Flow:            True


In [59]:
plt.figure()
plt.plot(data.self_similarity)
plt.plot(data.hypoxic_area)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [47]:
if 1:
    
#     file =  files[10]
#     print(file)
#     data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
    
#     data['positioned'] = data['positioning_frequency'].astype(str) + ', ' + data['repositioned'].astype(str)
#     data['positioned'] = data.positioned.apply(lambda x: x.replace('nan, nan','nan'))
    
    study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
    fs = hdr['fs']
    
    version = 'full'
    signals_available_plot = [x for x in signals_to_plot_full if x in data.columns]
    plot_routine()
    
    version = 'sub'
    signals_available_plot = [x for x in signals_to_plot_sub if x in data.columns]    
    plot_routine()
    
    version = 'resp'
    signals_available_plot = [x for x in signals_to_plot_resp if x in data.columns]    
    plot_routine()    

11
38.62102777777778


2020-06-01 20:00:00    Able to turn self, Turns self
2020-06-02 19:52:00    Able to turn self, Turns self
Name: positioned, dtype: object

In [17]:
signals_available_plot

['acc_ai_10sec',
 'apnea_pred_ro_a3',
 'apnea_pred_va_a3',
 'movavg_0_5s_unscaled',
 'rr_10s_smooth',
 'edw_respirations',
 'ventilation_10s_smooth',
 'instability_index_30sec',
 'instability_index_2min',
 'stage_pred_comb_breath_activity_1',
 'stage_pred_amendsumeffort',
 'hr',
 'spo2',
 'nbps',
 'nbpd',
 'positioned']

'/media/ssd2/Covid19_Respiration/Data_Analysis/Combined_Data_2_Plots/covid_resp_015_6.html'

In [11]:
plot?

Signature:
plot(
    figure_or_data,
    show_link=False,
    link_text='Export to plot.ly',
    validate=True,
    output_type='file',
    include_plotlyjs=True,
    filename='temp-plot.html',
    auto_open=True,
    image=None,
    image_filename='plot_image',
    image_width=800,
    image_height=600,
    config=None,
    include_mathjax=False,
    auto_play=True,
    animation_opts=None,
)
Docstring:
Create a plotly graph locally as an HTML document or string.

Example:
```
from plotly.offline import plot
import plotly.graph_objs as go

plot([go.Scatter(x=[1, 2, 3], y=[3, 2, 6])], filename='my-graph.html')
# We can also download an image of the plot by setting the image parameter
# to the image format we want
plot([go.Scatter(x=[1, 2, 3], y=[3, 2, 6])], filename='my-graph.html',
     image='jpeg')
```
More examples below.

figure_or_data -- a plotly.graph_objs.Figure or plotly.graph_objs.Data or
                  dict or list that describes a Plotly graph.
                  See htt

In [141]:
for file in files:
    print(file)
    data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
    study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
    fs = hdr['fs']
    
    signals_available_plot = [x for x in signals_to_plot if x in data.columns]
    
    # PLOT (html version)
    title = study_id

    fig = icu_sleep_plot(data[signals_available_plot], trace_linewidth=1, labelfontsize=12, legend_position = [0.2, 1.1], legend_font_size=12)
    fig.update_layout(title=title)
    plot(fig, filename = os.path.join(plots_dir, title+'.html'), auto_open=False) # save_path.replace('FILEFORMAT','HTML')+'.html'
    del fig

cov_resp_001.h5


KeyboardInterrupt: 

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/core/frame.py:4223: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

